# Computer Vision: From Pixels to Intelligence

**A hands-on journey through the foundations of computer vision (1960s–2000s)**

Before deep learning, computer vision was built by hand — mathematicians, geometricians, and signal processors who taught machines to "see" using calculus, linear algebra, and a deep understanding of how light interacts with the world.

In this notebook, we'll walk through the classical CV pipeline step by step. You'll write the code, run the operations, and see how each building block adds a layer of understanding.

> **The story:** We start with the simplest question — *what is an image to a computer?* — and build up to feature matching and image stitching. By the end, you'll understand the foundations that every modern CV system (including neural networks) still rests on.

## Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch  # Needed by timm/diffusers

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 14


## Generating Synthetic Test Images

We create test images using NumPy — self-contained, no external files needed.

> **Why synthetic?** In the 1960s–1980s, researchers worked with carefully controlled lab images — shaded boxes, geometric patterns, line drawings. Algorithms had to be clever, not data-hungry.

In [ ]:
def create_colorful_scene(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:, :] = (100, 150, 255)  # BGR background
    cv2.rectangle(img, (30, 30), (120, 100), (0, 0, 255), -1)  # Red rect
    cv2.circle(img, (180, 80), 40, (0, 255, 0), -1)  # Green circle
    pts = np.array([[size//2, 20], [size//2-50, size//2+20],
                    [size//2+50, size//2+20]], np.int32).reshape((-1,1,2))
    cv2.fillPoly(img, [pts], (0, 255, 255))  # Yellow triangle
    cv2.rectangle(img, (160, 140), (230, 220), (255,255,255), 3)  # White square
    cv2.rectangle(img, (170, 150), (220, 210), (0,0,0), -1)  # Black inner
    return img

def create_edge_scene(size=256):
    img = np.zeros((size, size), dtype=np.uint8)
    img[60:80, :] = 200
    img[140:160, :] = 200
    img[:, 60:80] = 200
    img[:, 160:180] = 200
    noise = np.random.randint(0, 30, (size, size), dtype=np.uint8)
    img = cv2.bitwise_or(img, noise)
    return img

def create_corner_scene(size=256):
    img = np.zeros((size, size), dtype=np.uint8)
    for i in range(0, size, 64):
        for j in range(0, size, 64):
            cv2.rectangle(img, (j+5, i+5), (j+55, i+55), 200, 2)
    cv2.line(img, (100, 100), (160, 100), 255, 3)
    cv2.line(img, (160, 100), (160, 160), 255, 3)
    return img

color_scene = create_colorful_scene()
edge_scene = create_edge_scene()
corner_scene = create_corner_scene()

print(f"Color scene: {color_scene.shape} {color_scene.dtype}")
print(f"Edge scene:  {edge_scene.shape} {edge_scene.dtype}")
print(f"Corner scene: {corner_scene.shape} {corner_scene.dtype}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(color_scene)
axes[0].set_title("Colorful Scene (BGR)")
axes[0].axis('off')
axes[1].imshow(edge_scene, cmap='gray')
axes[1].set_title("Edge Scene (Grayscale)")
axes[1].axis('off')
axes[2].imshow(corner_scene, cmap='gray')
axes[2].set_title("Corner Scene (Grayscale)")
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 1. What Is an Image? — From Photons to Numbers

To a computer, an image is a **discrete sampling of a continuous light field**. Understanding this transformation — from physical photons to a NumPy array — is the foundation of all computer vision.

### 1.1 Image Formation: The Pinhole Camera Model

Light reflects off objects in the world, travels through a lens, and strikes an image sensor. The **pinhole camera model** approximates this as a linear projection:

$$\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} \sim K \begin{bmatrix} R | t \end{bmatrix} \begin{bmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{bmatrix}$$

where $K$ is the **intrinsic matrix** (focal length, principal point, skew) and $[R|t]$ is the **extrinsic matrix** (rotation and translation of the camera in world coordinates). This is a **perspective projection** — 3D points are projected onto a 2D plane, losing depth information. This is the **fundamental ambiguity of vision**: infinitely many 3D worlds can produce the same 2D image.

### 1.2 Sampling and Quantization

The continuous image $I(x, y)$ is converted to a digital image through two steps:

1. **Sampling** (spatial discretization): The continuous image is sampled on a rectangular grid with spacing $\Delta_x, \Delta_y$. The digital image is $I[m\Delta_x, n\Delta_y] = I_d[m, n]$.
2. **Quantization** (amplitude discretization): Each sample's intensity is mapped to a discrete level. For 8-bit images: $q = \lfloor I / I_{\max} \cdot 255 \rfloor$.

**Nyquist-Shannon Sampling Theorem:** To avoid **aliasing** (artifacts from undersampling), the sampling frequency must be at least **twice the highest frequency** present in the image. This is why we blur (low-pass filter) before downsampling — to remove high frequencies that would alias.

**Quantization noise:** With $L$ quantization levels, the quantization error is uniformly distributed in $[-\Delta/2, \Delta/2]$ where $\Delta = I_{\max}/L$. The SNR is $SNR = 6.02b + 1.77$ dB where $b = \log_2 L$ bits per sample.

### 1.3 Color Image Formation: The Bayer Filter

Color sensors don't measure RGB directly. Each pixel has a **Bayer filter** — a mosaic of red, green, and blue filters (pattern: RG/GB). The camera performs **demosaicing** to interpolate the full RGB value at each pixel from its neighbors. This is itself an image processing problem — poor demosaicing causes color moiré artifacts.

### 1.4 Image Operations as Linear Algebra

Every image operation is a **transformation** on the array:

| Operation | Mathematical Form | Type |
|-----------|------------------|------|
| Brightness | $I'(i,j) = I(i,j) + b$ | Translation |
| Contrast | $I'(i,j) = \alpha \cdot I(i,j)$ | Scaling |
| Cropping | $I' = I[i_1:i_2, j_1:j_2]$ | Sub-sampling |
| Rotation | $I'(x,y) = I(x\cos\theta + y\sin\theta, -x\sin\theta + y\cos\theta)$ | Affine |
| Blurring | $I' = I * K$ (convolution) | Linear filter |

**Why parallelism?** Each pixel operation is **embarrassingly parallel** — pixel $(i,j)$ doesn't depend on pixel $(k,l)$. This makes GPUs ideal for image processing. Convolution is the exception: each output pixel depends on a local neighborhood, introducing **data dependency** within a kernel window.

In [ ]:
print("=== Grayscale ===")
print(f"Shape: {edge_scene.shape}  # (height, width)")
print(f"Range: [{edge_scene.min()}, {edge_scene.max()}]")
print(f"\nFirst 5x5 pixel block:")
print(edge_scene[:5, :5])

print("\n=== Color ===")
print(f"Shape: {color_scene.shape}  # (height, width, channels)")
print(f"Pixel at (0,0): BGR = {color_scene[0, 0]}")


### Image Operations as Linear Transformations

Every image operation is a **mathematical transformation** on the array. Understanding the algebra reveals what each operation can and cannot do:

| Operation | Mathematical Form | Geometric Meaning |
|-----------|------------------|-------------------|
| Brightness | $I'(i,j) = I(i,j) + b$ | Translation in intensity space |
| Contrast | $I'(i,j) = \alpha \cdot I(i,j)$ | Scaling in intensity space |
| Histogram equalization | $I' = \text{CDF}^{-1}(I)$ | Nonlinear remapping of intensities |
| Cropping | $I' = I[i_1:i_2, j_1:j_2]$ | Sub-sampling (spatial) |
| Rotation | $I'(x,y) = I(x\cos\theta + y\sin\theta, -x\sin\theta + y\cos\theta)$ | Affine transform |
| Blurring | $I' = I * K$ (convolution) | Linear low-pass filtering |

**Convolution** is the most important operation in CV. For a discrete image $I$ and kernel $K$ of size $(2k+1) \times (2k+1)$:

$$(I * K)[i,j] = \sum_{m=-k}^{k} \sum_{n=-k}^{k} I[i-m, j-n] \cdot K[m,n]$$

This is a **weighted sum** — each output pixel is a linear combination of its neighborhood. The kernel $K$ encodes the **prior** about what we're looking for (edges, blobs, textures).

**Why parallelism?** Pointwise operations (brightness, contrast, histogram equalization) are **embarrassingly parallel** — pixel $(i,j)$ doesn't depend on pixel $(k,l)$. Convolution introduces **data dependency** within the kernel window, but the window is small and fixed, making it still highly parallelizable on GPUs.

In [ ]:
# Brightness
brighter = np.clip(edge_scene.astype(float) * 1.5, 0, 255).astype(np.uint8)
# Crop
cropped = color_scene[40:200, 40:200]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(edge_scene, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')
axes[1].imshow(brighter, cmap='gray')
axes[1].set_title("Brightness x 1.5")
axes[1].axis('off')
axes[2].imshow(cropped)
axes[2].set_title("Cropped [40:200, 40:200]")
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 2. Color Spaces — Decomposing Light into Meaningful Components

RGB mixes **brightness** (illumination) and **color** (reflectance) together. If lighting changes from warm indoor to cool outdoor, all three RGB channels change — even if the object itself hasn't. This is the core problem that color space decomposition solves.

> **Historical note:** In the 1930s, the CIE (Commission Internationale de l'Éclairage) defined color spaces based on human psychophysics — experiments measuring how humans perceive color. In the 1970s–1980s, CV researchers realized these human-perception-based spaces were far more useful than RGB for segmentation and robustness. The key insight: **separate illumination from reflectance**.

### 2.1 The Problem with RGB

RGB is a **device-dependent** color space — it describes how a display emits light, not how humans perceive it. The three channels are **correlated**: changing illumination changes all three simultaneously. For a sunlight image vs. an incandescent image of the same red apple, the RGB values differ significantly even though the apple's **surface reflectance** (its true color) is identical.

Mathematically, the observed RGB at pixel $(i,j)$ is:

$$R_{\text{observed}}(i,j) = R_{\text{illumination}}(i,j) \odot R_{\text{reflectance}}(i,j)$$

where $\odot$ is element-wise multiplication. We want to recover $R_{\text{reflectance}}$ — the property that is invariant to lighting.

### 2.2 HSV: Hue, Saturation, Value

HSV separates color into three perceptually meaningful components via a **cylindrical transformation** of RGB:

- **Hue ($H$):** The dominant wavelength — "what color" (angle on the color circle: 0°=red, 120°=green, 240°=blue). Computed as the angle of the RGB vector projected onto the chromaticity plane.
- **Saturation ($S$):** The purity of the color — how much gray is mixed in. $S = 0$ means grayscale; $S = 1$ means fully saturated (no gray).
- **Value ($V$):** The overall brightness — the maximum of the three RGB channels.

The transformation is **nonlinear** — it maps the RGB cube into a hexagonal cone. Hue is computed as:

$$H = \arccos\left(\frac{\frac{1}{2}[(R-G) + (R-B)]}{\sqrt{(R-G)^2 + (R-B)(G-B)}}\right)$$

with quadrant adjustment. This angle is **invariant to brightness** — rotate the RGB vector around the grayscale axis and hue stays constant.

### 2.3 CIELAB: Perceptually Uniform Color Space

CIELAB (1976) was designed to be **perceptually uniform**: equal distances in LAB space correspond to equal perceived color differences. This is achieved through a **nonlinear compression** of the CIE XYZ tristimulus values:

$$L^* = 116 \cdot f(Y/Y_n) - 16 \quad \text{(Lightness: 0 = black, 100 = white)}$$
$$a^* = 500 \cdot [f(X/X_n) - f(Y/Y_n)] \quad \text{(Red-Green axis)}$$
$$b^* = 200 \cdot [f(Y/Y_n) - f(Z/Z_n)] \quad \text{(Yellow-Blue axis)}$$

where $f(t) = t^{1/3}$ for $t > 0.008856$, else $f(t) = 7.787t + 16/116$.

**Why CIELAB matters for CV:**
- The $L^*$ channel is **illumination-invariant** for segmentation (unlike grayscale)
- The Euclidean distance $\Delta E = \sqrt{\Delta L^{*2} + \Delta a^{*2} + \Delta b^{*2}}$ corresponds to **perceived color difference**
- The $a^*$ and $b^*$ channels are independent of brightness, making them ideal for color-based segmentation under varying lighting

### 2.4 Choosing the Right Color Space

| Task | Best Space | Why |
|------|-----------|-----|
| Display / capture | RGB / BGR | Device-native |
| Color segmentation | HSV (Saturation) | Brightness-independent thresholding |
| Perceptual similarity | CIELAB | Uniform distance metric |
| Edge detection | L channel of Lab | Separates edges from illumination changes |
| Skin detection | YCrCb or Lab | Skin occupies a tight cluster in Cr-Cb plane |

In [ ]:
bgr = color_scene
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

h, s, v = cv2.split(hsv)
l, a, b_ch = cv2.split(lab)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(bgr)
axes[0, 0].set_title("BGR (OpenCV default)")
axes[0, 0].axis('off')
axes[0, 1].imshow(rgb)
axes[0, 1].set_title("RGB")
axes[0, 1].axis('off')
axes[0, 2].imshow(gray, cmap='gray')
axes[0, 2].set_title("Grayscale (luminance only)")
axes[0, 2].axis('off')

axes[1, 0].imshow(h, cmap='viridis')
axes[1, 0].set_title("Hue (color type)")
axes[1, 0].axis('off')
axes[1, 1].imshow(s, cmap='viridis')
axes[1, 1].set_title("Saturation (color intensity)")
axes[1, 1].axis('off')
axes[1, 2].imshow(l, cmap='viridis')
axes[1, 2].set_title("Lightness (CIELAB)")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### Key Insight: Saturation as an Illumination-Invariant Feature

The **Saturation** channel tells you where colored objects are, regardless of brightness. The **Hue** channel tells you *what color* something is, independent of lighting.

**Mathematical justification:** Saturation is defined as $S = 1 - \frac{\min(R,G,B)}{\max(R,G,B)}$. Under a multiplicative illumination change $I' = \alpha \cdot I$, both the min and max scale by $\alpha$, so $S' = 1 - \frac{\alpha \cdot \min}{\alpha \cdot \max} = S$. **Saturation is provably invariant to brightness scaling.**

This is why thresholding the saturation channel is a robust segmentation strategy: it works under shadows, varying exposure, and different light sources — as long as the relative channel ratios are preserved.

> **Hands-on:** Threshold the saturation channel to segment colored shapes:

In [ ]:
# Hands-on: Try different threshold values
threshold = 50  # <-- Try changing this!
mask = (s > threshold).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(s, cmap='viridis')
axes[0].set_title(f"Saturation (threshold={threshold})")
axes[0].axis('off')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Binary Mask")
axes[1].axis('off')
axes[2].imshow(bgr)
axes[2].set_title("Original")
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 3. Edge Detection — Finding Boundaries

Edges are where pixel intensity changes sharply — object boundaries, surface discontinuities, shadows, and specular highlights. They are the **lowest-level primitive** for visual understanding: from edges, we build contours, shapes, and ultimately object recognition.

> **Historical timeline:**
> - **1960s:** Edwin Land's RETINEX theory showed that edges (relative intensity changes) matter more to human vision than absolute brightness. The brain computes **contrast**, not absolute luminance.
> - **1970s:** David Marr's theory of vision: edge detection is the first computational step in visual understanding. The primal sketch represents the world as edges, blobs, and lines.
> - **1983:** Marr and Hildreth formalized the **theory of edge detection**: compute the Laplacian, find zero-crossings. This connected edge detection to band-pass filtering.
> - **1986:** John Canny published the optimal edge detector with three rigorous criteria: good detection, good localization, minimal response. This remains the gold standard.

### 3.1 The Gradient: Edges as First Derivatives

An edge is a **discontinuity in intensity**. Mathematically, we detect it by computing the **gradient** of the intensity function $I(x, y)$:

$$\nabla I = \begin{bmatrix} \frac{\partial I}{\partial x} \\ \frac{\partial I}{\partial y} \end{bmatrix} = \begin{bmatrix} G_x \\ G_y \end{bmatrix}$$

The gradient points in the direction of **maximum intensity increase**, and its magnitude tells us the **strength** of the edge:

$$|\nabla I| = \sqrt{G_x^2 + G_y^2}, \quad \theta = \arctan2(G_y, G_x)$$

where $\theta$ is the **edge direction** (perpendicular to the edge orientation).

**Why convolution?** Derivatives are computed by convolving with derivative kernels. The Sobel operator combines **smoothing** (to reduce noise) with **differentiation** (to find edges):

$$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * I, \quad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * I$$

The middle column has weight 2 because **Gaussian smoothing** in the y-direction is approximated by the kernel $[1, 2, 1]$ (binomial coefficients). This is not arbitrary — it's a separable approximation of a Gaussian derivative.

### 3.2 Why Sobel Isn't Enough: The Trade-offs

Sobel has three fundamental problems:
1. **Noise sensitivity:** The gradient amplifies high-frequency noise. A single noisy pixel can create a spurious edge.
2. **Thick edges:** The gradient magnitude is high across a *region*, not just at the edge center. Edges are 3-5 pixels wide.
3. **No thresholding strategy:** How do you choose the threshold? Too low → noise edges. Too high → miss real edges.

### 3.3 Canny's Three Optimality Criteria (1986)

Canny formalized edge detection as an **optimization problem** with three criteria:

1. **Good Detection:** Minimize the probability of missing real edges AND false responses to non-edges. Maximize the signal-to-noise ratio (SNR).

2. **Good Localization:** The detected edge position should be as close as possible to the true edge position. Minimize the variance of the edge position error.

3. **Minimal Response:** A single edge should produce a single response. Minimize multiple responses to the same edge.

Canny proved that the **optimal operator** for edge detection is proportional to the **first derivative of a Gaussian**:

$$h_{\text{optimal}}(x) \propto \frac{d}{dx} e^{-x^2/(2\sigma^2)} = -\frac{x}{\sigma^2} e^{-x^2/(2\sigma^2)}$$

This is the **Gaussian derivative** — it combines Gaussian smoothing (for noise rejection) with differentiation (for edge detection). The parameter $\sigma$ controls the **trade-off**: larger $\sigma$ = more smoothing = better noise rejection but worse localization.

### 3.4 The Canny Pipeline: Approximating Optimality

The full Canny algorithm implements these principles in 5 steps:

1. **Gaussian blur** ($\sigma \approx 1$): Approximate the optimal smoothing. Reduces high-frequency noise before differentiation.
2. **Gradient computation** (Sobel approximation): Compute $G_x$, $G_y$, magnitude, and direction. The Sobel is a fast approximation of the Gaussian derivative.
3. **Non-maximum suppression (NMS)**: Thin edges to 1 pixel. For each pixel, compare its gradient magnitude with its two neighbors in the gradient direction. If it's not a local maximum, suppress it to zero. This enforces **minimal response**.
4. **Double thresholding**: Classify pixels as strong edges ($|\nabla I| > T_{\text{high}}$), weak edges ($T_{\text{low}} < |\nabla I| < T_{\text{high}}$), or non-edges.
5. **Hysteresis thresholding**: Keep strong edges. Keep weak edges **only if connected to a strong edge**. This enforces **good detection** (real edges that fade are kept) while rejecting noise (isolated weak responses are discarded).

In [ ]:
# Build Sobel kernels from scratch
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

# Convolution via filter2D
grad_x = cv2.filter2D(edge_scene.astype(np.float32), -1, sobel_x)
grad_y = cv2.filter2D(edge_scene.astype(np.float32), -1, sobel_y)
magnitude = np.sqrt(grad_x**2 + grad_y**2)
direction = np.arctan2(grad_y, grad_x)

mag_display = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(edge_scene, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')
axes[1].imshow(grad_x, cmap='coolwarm')
axes[1].set_title("Gx (vertical edges)")
axes[1].axis('off')
axes[2].imshow(grad_y, cmap='coolwarm')
axes[2].set_title("Gy (horizontal edges)")
axes[2].axis('off')
axes[3].imshow(mag_display, cmap='gray')
axes[3].set_title("Magnitude sqrt(Gx^2 + Gy^2)")
axes[3].axis('off')
plt.tight_layout()
plt.show()


### The Canny Edge Detector (1986) — A Rigorous Pipeline

Canny solved Sobel's problems with a mathematically-grounded 5-step pipeline. Each step addresses a specific failure mode:

1. **Gaussian blur** — reduces high-frequency noise that would create false gradient responses. The Gaussian is optimal because it achieves the Heisenberg uncertainty principle bound — the best possible joint spatial-frequency localization.

2. **Gradient computation** — Sobel kernels approximate the Gaussian derivative. This gives both edge strength ($|\nabla I|$) and edge orientation ($\theta$).

3. **Non-maximum suppression (NMS)** — thicks edges to 1 pixel. For each pixel, interpolate the gradient magnitudes of its two neighbors along the gradient direction. If the current pixel is not the maximum, set it to zero. This enforces the **minimal response** criterion.

4. **Double threshold** — separates strong edges (high confidence) from weak edges (possibly real, possibly noise). Pixels with $|\nabla I| > T_{\text{high}}$ are **definitely edges**. Pixels with $|\nabla I| < T_{\text{low}}$ are **definitely not edges**. Pixels in between are **conditional edges**.

5. **Hysteresis tracking** — connects weak edges to strong edges via 8-connectivity. A weak edge is kept if there exists a path of connected weak/conditional edges leading to a strong edge. This is essentially **graph connectivity** — weak edges that are "anchored" to strong edges are real; isolated weak edges are noise.

In [ ]:
# Full Canny pipeline
blurred = cv2.GaussianBlur(edge_scene, (5, 5), 1.5)
canny = cv2.Canny(edge_scene, 50, 150)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(edge_scene, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')
axes[1].imshow(blurred, cmap='gray')
axes[1].set_title("1. Gaussian Blur")
axes[1].axis('off')
gx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0)
gy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1)
mag = np.sqrt(gx**2 + gy**2)
axes[2].imshow(cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), cmap='gray')
axes[2].set_title("2. Gradient (thick edges)")
axes[2].axis('off')
axes[3].imshow(canny, cmap='gray')
axes[3].set_title("3. Canny (thin, clean)")
axes[3].axis('off')
plt.tight_layout()
plt.show()


### Hysteresis: The Two-Threshold Strategy

The choice of a single edge threshold is a **fundamental dilemma**:
- **High threshold:** Clean edges, but real edges that fade (low contrast regions) are broken.
- **Low threshold:** Continuous edges, but many noise edges appear.

**Hysteresis** (from physics: the dependence of a system's state on its history) solves this with two thresholds:

- **$T_{\text{high}}$** (e.g., 150): Strong edges only — high confidence boundaries
- **$T_{\text{low}}$** (e.g., 50): Weak edges — possible boundaries

A weak edge survives **only if connected to a strong edge** via 8-connectivity. This is implemented as a graph traversal (DFS or BFS) from strong edge pixels, following weak edge neighbors.

**Why "hysteresis"?** The term comes from the physics of systems with memory — whether a weak edge is kept depends on whether it has been "exposed" to a strong edge (the system's history). An isolated weak edge is discarded; a weak edge adjacent to a strong edge is preserved.

**Choosing thresholds empirically:** A common heuristic is $T_{\text{low}} = 0.4 \cdot T_{\text{high}}$. The absolute values depend on the gradient magnitude range. For noisy images, increase both. For low-contrast images, decrease both.

> **Hands-on:** Experiment with thresholds:

In [ ]:
# Hands-on: Try different threshold pairs
low, high = 30, 100  # <-- Experiment!
edges = cv2.Canny(edge_scene, low, high)

plt.figure(figsize=(10, 4))
plt.imshow(edges, cmap='gray')
plt.title(f"Canny (low={low}, high={high})")
plt.axis('off')
plt.tight_layout()
plt.show()

# Try: low=0, high=255 (everything is an edge)
# Try: low=200, high=255 (only strongest edges)


## 4. Corner Detection — Where Edges Meet

Corners are intersections where intensity changes in **all directions**. They are far more informative than edges because they are **stable, repeatable, and localizable** across viewpoints — making them the cornerstone of feature-based computer vision.

> **Historical timeline:**
> - **1988:** Harris & Stephens publish the corner detector ("A Combined Corner and Edge Detector"). This is the first practical, real-time corner detector.
> - **1989:** Tomasi & Kanade prove that corners detected by their **good features to track** criterion are equivalent to Harris corners with a specific parameter choice.
> - **1999:** David Lowe discovers SIFT — scale-invariant keypoints that work across different scales and viewpoints.
> - **2004:** SIFT wins the International Competition on Feature-detection.
> - **2011:** ORB is proposed as a fast, patent-free alternative to SIFT.

### 4.1 The Structure Matrix: Measuring Local Intensity Variation

Consider a small window $W$ centered at pixel $(x, y)$. The intensity change when the window is shifted by $(u, v)$ is:

$$E(u, v) = \sum_{(x,y) \in W} w(x,y) \cdot [I(x+u, y+v) - I(x,y)]^2$$

where $w(x,y)$ is a windowing function (usually Gaussian). Using the **first-order Taylor approximation** $I(x+u, y+v) \approx I(x,y) + I_x \cdot u + I_y \cdot v$:

$$E(u, v) \approx \sum_{(x,y) \in W} w(x,y) \cdot (I_x \cdot u + I_y \cdot v)^2 = \begin{bmatrix} u & v \end{bmatrix} M \begin{bmatrix} u \\ v \end{bmatrix}$$

where $M$ is the **structure matrix** (also called the second-moment matrix):

$$M = \sum_{(x,y) \in W} w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix} = \begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}$$

$M$ is a **2×2 symmetric positive semi-definite matrix**. Its eigenvalues $\lambda_1, \lambda_2$ (with $\lambda_1 \geq \lambda_2 \geq 0$) tell us the **shape of the intensity variation** in the window.

### 4.2 Eigenvalue Classification: The Geometry of Corners

The eigenvalues of $M$ classify every pixel into one of three categories:

| Eigenvalues | Geometry | Interpretation |
|-------------|----------|----------------|
| $\lambda_1 \approx 0$, $\lambda_2 \approx 0$ | Flat region | No intensity change in any direction |
| $\lambda_1 \gg 0$, $\lambda_2 \approx 0$ | Edge | Strong change in one direction, flat in the other |
| $\lambda_1 \gg 0$, $\lambda_2 \gg 0$ | **Corner** | Strong change in **all** directions |

**The key insight:** Corners are windows where intensity changes significantly in **all directions** — both eigenvalues are large. This makes them **repeatable**: the same corner appears in different views because the local structure is distinctive.

### 4.3 The Harris Corner Response Function

Computing eigenvalues is expensive. Harris and Stephens derived a **scoring function** that approximates the eigenvalue analysis without explicit decomposition:

$$R = \det(M) - k \cdot \text{trace}(M)^2 = \lambda_1 \lambda_2 - k(\lambda_1 + \lambda_2)^2$$

where $k \approx 0.04-0.06$ is the **Harris sensitivity parameter**.

**Why this works:**
- If $\lambda_1, \lambda_2 \approx 0$: $R \approx 0$ (flat)
- If $\lambda_1 \gg \lambda_2 \approx 0$: $R \approx \lambda_1 \cdot 0 - k \cdot \lambda_1^2 < 0$ (edge, negative)
- If $\lambda_1, \lambda_2 \gg 0$: $R > 0$ (corner, positive)

The parameter $k$ controls the **edge rejection**: smaller $k$ makes the detector more sensitive to corners but also more prone to false positives. The trace term penalizes windows with large eigenvalues in only one direction (edges), while the determinant term rewards windows with large eigenvalues in both directions (corners).

### 4.4 Non-Maximum Suppression

After computing the Harris response $R(x,y)$ for all pixels, we perform **non-maximum suppression**: keep only local maxima of the response. This ensures that corners are detected as **single points**, not clusters. The standard approach: dilate the response map, then keep only pixels where the original response equals the dilated maximum.

In [ ]:
# Harris corner detection
harris = cv2.cornerHarris(edge_scene.astype(np.float32), blockSize=2, ksize=3, k=0.04)
harris_dilated = cv2.dilate(harris, None)
threshold = 0.01 * harris_dilated.max()
corners = harris_dilated > threshold

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(corner_scene, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')
axes[1].imshow(harris, cmap='hot')
axes[1].set_title("Harris Response (bright = corner)")
axes[1].axis('off')
axes[2].imshow(corner_scene, cmap='gray')
y_coords, x_coords = np.where(corners)
for x, y in zip(x_coords, y_coords):
    axes[2].plot(x, y, 'r+', markersize=8, markeredgewidth=2)
axes[2].set_title(f"Detected Corners ({len(x_coords)} found)")
axes[2].axis('off')
plt.tight_layout()
plt.show()


### Why Corners Are the Gold Standard for Feature Detection

Corners are **repeatable features** — the same physical corner can be found in two images taken from different viewpoints. This repeatability enables virtually all feature-based CV:

- **Image stitching:** Match corners between overlapping images → estimate **homography** → warp images into a panorama.
- **3D reconstruction:** Match corners across multiple views → **triangulate** 3D points → recover scene geometry (structure from motion).
- **Object tracking:** Track corners across video frames → estimate motion and occlusion.
- **Robot localization:** Match current image corners to a pre-built map → determine camera pose (PnP problem).
- **Image retrieval:** Describe regions around corners → match images by shared features.

**What makes a good feature?** (Lowe's criteria for SIFT, but applicable generally):
1. **Detectability:** Can be found reliably across views
2. **Distinctiveness:** The local descriptor uniquely identifies the feature
3. **Quantity:** Enough features to cover the scene
4. **Efficiency:** Can be computed in real-time
5. **Invariance:** Robust to scale, rotation, illumination changes

Harris satisfies criteria 1-3 but **not** 5 (no scale or rotation invariance). SIFT and ORB extend it to satisfy all five.

## 5. Contours & Shapes — From Pixels to Topology

Binary images (edges, thresholds) contain **connected components** — groups of pixels that form coherent shapes. **Contour tracing** extracts the boundaries of these components, converting pixel-level data into geometric primitives.

> **Historical note:** The Suzuki topological contour tracing algorithm (1985) is the foundation of OpenCV's `findContours`. It classifies contours into a **hierarchy** based on nesting relationships (one shape inside another), capturing the topology of the scene. Pure algorithmic geometry — no learning, no statistics, just careful case analysis.

### 5.1 Contour Tracing: The Boundary-Following Algorithm

The problem: given a binary image, find the boundary pixels of each connected component. The Suzuki algorithm uses **double-loop tracing**:

1. Find the first white pixel (boundary of an outer contour).
2. Follow the boundary in a clockwise direction, always keeping the interior on the right.
3. When the boundary closes, the outer contour is complete.
4. Search for the next untraced white pixel (could be a hole boundary).
5. Repeat, building a **parent-child hierarchy** of contours.

The hierarchy has two levels:
- **External contours:** Not enclosed by any other contour (outer boundaries).
- **Hole contours:** Enclosed by an external contour (holes, windows, etc.).

### 5.2 Chain Codes and Polygonal Approximation

A contour can be encoded as a **chain code** (Freeman chain code): a sequence of directions (0-7) describing each step along the boundary. This compresses the contour representation.

**Polygonal approximation** (Douglas-Peucker algorithm): simplify a contour to a polygon with $n$ vertices. Iteratively:
1. Connect the first and last points with a straight line.
2. Find the point farthest from this line.
3. If the distance exceeds a threshold $\epsilon$, keep this point as a vertex and recurse.
4. Otherwise, discard it.

The parameter $\epsilon$ controls the **simplification level**: larger $\epsilon$ = fewer vertices = coarser approximation. This is used in `cv2.approxPolyDP` to classify shapes:
- 3 vertices → triangle
- 4 vertices → rectangle/quad
- >6 vertices → circle/ellipse (rounded)

### 5.3 Shape Descriptors

Once we have a contour, we can compute **invariant descriptors**:

| Descriptor | Formula | Invariance |
|-----------|---------|------------|
| Area | $\frac{1}{2}\sum(x_i y_{i+1} - x_{i+1} y_i)$ (Shoelace) | Translation |
| Perimeter | $\sum \sqrt{(x_{i+1}-x_i)^2 + (y_{i+1}-y_i)^2}$ | Translation, rotation |
| Bounding box | $\min x, \min y, \max x, \max y$ | Translation |
| Extent | $\text{area} / (\text{bbox area})$ | Translation, rotation, scale |
| Solidity | $\text{area} / \text{convex hull area}$ | Translation, rotation, scale |
| Circularity | $4\pi \cdot \text{area} / \text{perimeter}^2$ | Translation, rotation, scale |

**Circularity = 1** for a perfect circle, **< 1** for any other shape. A square has circularity $\pi/4 \approx 0.785$. This is a simple but powerful shape classifier.

In [ ]:
# Find contours in saturation mask
binary = (s > 50).astype(np.uint8) * 255
contours, hierarchy = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

contour_img = bgr.copy()
cv2.drawContours(contour_img, contours, -1, (255, 255, 0), 2)

print(f"Found {len(contours)} contours")
print(f"{'Idx':<4} {'Area':<10} {'Perimeter':<10} {'Bounding Box':<18} {'Shape'}")
print("-" * 60)
for i, c in enumerate(contours):
    area = cv2.contourArea(c)
    if area < 100:
        continue
    peri = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    approx = cv2.approxPolyDP(c, 0.04 * peri, True)
    n = len(approx)
    shape = {3: "Triangle", 4: "Rectangle"}.get(n, "Circle" if n > 6 else f"{n}-gon")
    print(f"{i:<4} {area:<10.0f} {peri:<10.1f} ({x},{y},{w}x{h})    {shape}")

plt.figure(figsize=(12, 6))
plt.imshow(contour_img)
plt.title(f"{len([c for c in contours if cv2.contourArea(c) > 100])} Shapes Detected")
plt.axis('off')
plt.tight_layout()
plt.show()


## 6. Feature Detection & Matching — The 2000s Golden Age

Harris corners are **point features** — but they fail when the same scene is viewed at different scales. A corner visible at one zoom level may not exist (or may be at a different pixel) at another scale. The breakthrough came with **scale-invariant feature detectors** that find the same points across multiple scales.

> **Timeline:**
> - **1999:** Lowe publishes SIFT (Scale-Invariant Feature Transform) — detects and describes keypoints that are invariant to scale, rotation, and partially invariant to illumination and viewpoint. This is the **most influential CV paper of the 2000s**.
> - **2006:** SURF (Speeded-Up Robust Features) uses integral images and box filters to approximate SIFT at 3× speed.
> - **2011:** ORB (Oriented FAST and Rotated BRIEF) combines the FAST keypoint detector with the BRIEF descriptor, creating a fast, patent-free alternative to SIFT.

### 6.1 Scale Space and the Difference of Gaussians (SIFT)

**The core idea:** Find points that are stable across scales. The scale space of an image is generated by convolving with Gaussians of increasing $\sigma$:

$$L(x, y, \sigma) = G(x, y, \sigma) * I(x, y)$$

where $G$ is a Gaussian kernel. The **Difference of Gaussians (DoG)** approximates the Laplacian of Gaussian and highlights regions of interest at each scale:

$$\text{DoG}(x, y, \sigma) = L(x, y, k\sigma) - L(x, y, \sigma)$$

Keypoints are **local extrema** in the DoG pyramid — points that are maxima or minima across both space and scale. This ensures that detected features are **scale-invariant**: the same physical point is found regardless of zoom level.

**Why Gaussian?** The Gaussian is the **unique** kernel that creates scale space without introducing new spurious extrema at different scales (Theorem by Lindeberg, 1994). No other smoothing kernel guarantees this property.

### 6.2 SIFT Descriptors: Orientation-Weighted Histograms

Once a keypoint is found at a specific scale and location, SIFT creates a **rotation-invariant descriptor**:

1. **Assign orientation:** Compute the gradient magnitude and direction in a neighborhood around the keypoint. Create a histogram of orientations. The dominant orientation(s) are assigned to the keypoint.
2. **Rotate to canonical orientation:** Rotate all gradients by the negative of the dominant orientation.
3. **Create 4×4 sub-regions:** Divide the neighborhood into 4×4 cells. Each cell stores an 8-bin orientation histogram.
4. **Result:** A 128-dimensional vector (4 × 4 × 8).

This descriptor is **robust** to small affine changes, illumination changes (it uses gradients, not absolute intensities), and noise (the histogram bins smooth over small variations).

### 6.3 ORB: FAST Keypoints + BRIEF Descriptors

ORB takes a different approach — **speed** without sacrificing too much quality:

**FAST (Features from Accelerated Segment Test):** A keypoint detector that is orders of magnitude faster than SIFT. It checks if a circle of 16 pixels around a candidate has enough consecutive pixels that are all brighter or darker than the center by a threshold. If so, it's a corner. No scale space — just fast corner detection at a single scale.

**BRIEF (Binary Robust Independent Elementary Features):** A descriptor that compares pairs of pixel intensities in a neighborhood:

$$b(x, y) = \begin{cases} 1 & \text{if } I(p) < I(q) \\ 0 & \text{otherwise} \end{cases}$$

The result is a **binary string** (e.g., 256 bits). Matching uses **Hamming distance** (count of differing bits), which is extremely fast on modern CPUs (single instruction).

**ORB's improvements over original BRIEF:**
- **Orientation:** Assigns a dominant orientation to each keypoint (like SIFT) and rotates the sampling pattern.
- **Optimized sampling:** The pixel pair locations $(p, q)$ are learned via a genetic algorithm to maximize variance and minimize correlation between bits.

### 6.4 Matching: From Raw Matches to Correspondences

The fundamental problem: given $N$ descriptors in image A and $M$ descriptors in image B, find the $K$ pairs that represent the **same physical point**.

**Brute-force matching** computes the distance between every pair — $O(N \cdot M)$ complexity. For binary descriptors, this uses Hamming distance. For SIFT (128-d float), this uses L2 distance.

**The ambiguity problem:** Many descriptors are similar. A uniform wall might have many similar gradient histograms. We need a way to **filter out ambiguous matches**.

In [ ]:
# ORB feature detection
orb = cv2.ORB_create(nfeatures=500)
kp1, desc1 = orb.detectAndCompute(edge_scene, None)
kp2, desc2 = orb.detectAndCompute(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), None)

print(f"Image 1: {len(kp1)} keypoints")
print(f"Image 2: {len(kp2)} keypoints")

img_kp = cv2.drawKeypoints(edge_scene, kp1, None,
    color=(0, 255, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(12, 6))
plt.imshow(img_kp)
plt.title(f"{len(kp1)} ORB Keypoints")
plt.axis('off')
plt.tight_layout()
plt.show()


### Lowe's Ratio Test (2004) — Filtering Ambiguous Matches

For each keypoint in image A, find the **two nearest** descriptors in image B (using k-NN search). Let $d_1$ be the distance to the best match and $d_2$ be the distance to the second-best match.

**The ratio test:** Keep the match only if:

$$d_1 < \rho \cdot d_2$$

where $\rho = 0.7$ (Lowe's recommended value).

**Why this works:** If the descriptor is **distinctive**, the best match will be much closer than the second-best (small ratio). If the descriptor is **ambiguous** (e.g., a uniform texture), the two nearest neighbors will be at similar distances (ratio close to 1). The ratio test filters out ambiguous matches while keeping distinctive ones.

**Mathematical intuition:** The ratio is **invariant to the absolute scale** of the descriptor space. It only depends on the relative separation of the nearest neighbors. This makes it robust to the choice of descriptor dimensionality and normalization.

**Further filtering:** After the ratio test, use **RANSAC** (see Section 7) to remove geometrically inconsistent matches. The ratio test removes ambiguous descriptors; RANSAC removes matches that don't agree on a geometric transformation.

In [ ]:
# BFMatcher with Hamming distance (binary descriptors)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
matches = bf.knnMatch(desc1, desc2, k=2)

# Lowe's ratio test
good = [m for m, n in matches if m.distance < 0.7 * n.distance]
print(f"Raw matches: {len(matches)} -> After ratio test: {len(good)} good matches")

match_img = cv2.drawMatches(edge_scene, kp1, cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), kp2,
    good[:50], flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(16, 8))
plt.imshow(match_img)
plt.title(f"Feature Matching: {len(good)} good matches")
plt.axis('off')
plt.tight_layout()
plt.show()


## 7. Image Stitching — The Classical CV Capstone

Image stitching (panorama creation) is where classical CV peaked — building something genuinely useful with nothing but geometry, linear algebra, and clever algorithms. No training, no data, no neural networks.

> **Historical note:** Panorama stitching was one of the first consumer CV applications. The first digital panorama was created by Brown and Lowe in the 1990s. By the 2000s, stitching was built into consumer software (Microsoft PhotoStory, Google Picasa). Under the hood: feature detection → matching → homography estimation → warping → blending.

### 7.1 The Full Pipeline

1. **Detect features** in both images (ORB, SIFT, etc.)
2. **Match features** (BFMatcher + Lowe's ratio test)
3. **Estimate homography** using RANSAC (robust to outlier matches)
4. **Warp** one image using the homography to align with the other
5. **Blend** the overlap region to remove visible seams

### 7.2 The Homography Matrix

A **homography** is a $3 \times 3$ matrix that describes the projective transformation between two views of a planar surface (or a scene where the camera rotates about its center):

$$\begin{bmatrix} x' \\ y' \\ w' \end{bmatrix} \sim H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}, \quad H = \begin{bmatrix} h_1 & h_2 & h_3 \\ h_4 & h_5 & h_6 \\ h_7 & h_8 & h_9 \end{bmatrix}$$

The transformation in pixel coordinates is:

$$x' = \frac{h_1 x + h_2 y + h_3}{h_7 x + h_8 y + h_9}, \quad y' = \frac{h_4 x + h_5 y + h_6}{h_7 x + h_8 y + h_9}$$

**Degrees of freedom:** $H$ has 8 degrees of freedom (scale-invariant: $H$ and $cH$ are equivalent). Each point correspondence $(x, y) \to (x', y')$ provides 2 equations. Therefore, we need **at least 4 point correspondences** to solve for $H$ (the "Direct Linear Transform").

### 7.3 RANSAC: Robust Estimation in the Presence of Outliers

After the ratio test, we still have **outlier matches** — incorrect correspondences that pass the ratio test by chance. A naive least-squares fit of $H$ would be dominated by these outliers.

**RANSAC** (Random Sample Consensus, Fischler & Bolles, 1981) works as follows:

1. **Sample:** Randomly select the minimum number of points (4) to compute a candidate homography $H_i$.
2. **Verify:** For each match, check if it is consistent with $H_i$ (i.e., the reprojection error $\|(x', y') - H_i(x, y)\| < T$). Count the inliers.
3. **Iterate:** Repeat $N$ times, keeping the model with the most inliers.
4. **Refit:** Recompute $H$ using all inliers from the best model (least-squares).

**Convergence guarantee:** If the probability of an inlier is $p$ and $s$ points are needed, the number of iterations required for at least one outlier-free sample with probability $1-p_{\text{fail}}$ is:

$$N = \frac{\log(1 - p_{\text{fail}})}{\log(1 - (1-w)^s)}$$

where $w$ is the estimated inlier ratio. For $w = 0.3$, $s = 4$, $p_{\text{fail}} = 0.99$: $N \approx 12$ iterations.

**Why RANSAC works:** It doesn't try to use all data — it tries to find the **subset** of data that agrees on a model. This makes it robust to arbitrarily many outliers (as long as there's at least one outlier-free sample).

In [ ]:
# Create overlapping panorama pair
def create_panorama_pair(size=512, overlap=150):
    scene = np.zeros((size, size + overlap, 3), dtype=np.uint8)
    for x in range(scene.shape[1]):
        t = x / scene.shape[1]
        scene[:, x] = (int(50+200*t), int(100+100*(1-t)), int(200-100*t))
    cv2.circle(scene, (100, size//2), 40, (0, 255, 0), -1)
    cv2.rectangle(scene, (size//2-30, size//2-30), (size//2+30, size//2+30), (0, 0, 255), -1)
    cv2.circle(scene, (size+overlap-100, size//2), 35, (255, 255, 0), -1)
    return scene[:, :size-overlap//2], scene[:, overlap//2:]

left, right = create_panorama_pair()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(left)
axes[0].set_title("Left Image")
axes[0].axis('off')
axes[1].imshow(right)
axes[1].set_title("Right Image")
axes[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# === STEP 1: Detect & match ===
orb = cv2.ORB_create(nfeatures=1000)
kp_l, desc_l = orb.detectAndCompute(cv2.cvtColor(left, cv2.COLOR_BGR2GRAY), None)
kp_r, desc_r = orb.detectAndCompute(cv2.cvtColor(right, cv2.COLOR_BGR2GRAY), None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING)
matches = bf.knnMatch(desc_l, desc_r, k=2)
good = [m for m, n in matches if m.distance < 0.7 * n.distance]
print(f"Matches: {len(good)}")

# === STEP 2: Homography via RANSAC ===
if len(good) >= 4:
    src_pts = np.float32([kp_l[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_r[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    print(f"Homography:\n{H}")
    print(f"Inliers: {np.sum(mask)} / {len(good)}")


In [ ]:
# === STEP 3: Warp & blend ===
if H is not None:
    h_l, w_l = left.shape[:2]
    h_r, w_r = right.shape[:2]
    
    # Warp right image
    stitched = cv2.warpPerspective(right, H, (w_l + w_r, h_l))
    stitched[0:h_l, 0:w_l] = left
    
    # Simple linear blend in overlap
    for x in range(max(0, w_l-150), min(w_l+w_r, w_l+150)):
        if x < w_l:
            lw = 1.0 - (x - max(0, w_l-150)) / 150.0
        else:
            lw = 0.0
        rw = 1.0 - lw
        stitched[0:h_l, x] = (lw * stitched[0:h_l, x] + rw * right[0:h_r, x-w_l]).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(left)
    axes[0].set_title("Left")
    axes[0].axis('off')
    axes[1].imshow(right)
    axes[1].set_title("Right")
    axes[1].axis('off')
    axes[2].imshow(stitched)
    axes[2].set_title("Stitched Panorama")
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()


### The Homography Matrix and Warping

$$\begin{bmatrix} x' \\ y' \\ w' \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

OpenCV uses **RANSAC** (Random Sample Consensus): randomly sample 4 point pairs, compute candidate homography via DLT (Direct Linear Transform), keep the one with the most inliers. Robust to outlier matches.

**Warping:** Once $H$ is estimated, we use `cv2.warpPerspective` to resample the right image into the coordinate frame of the left image. This involves **inverse warping**: for each pixel in the output, compute the corresponding pixel in the source image using $H^{-1}$, then interpolate (bilinear or bicubic) to handle non-integer coordinates.

**Blending:** Simple linear blending in the overlap region creates visible seams due to exposure differences. Better approaches:
- **Multi-band (Pyramid) blending:** Decompose both images into a Laplacian pyramid, blend the lowest levels with a Gaussian window, and reconstruct. This blends at multiple frequencies, eliminating visible seams.
- **Feathered blending:** The approach used in the code — a linear gradient in the overlap region. Simple but effective for images with similar exposure.

> **This is the capstone of classical CV:** Feature detection → matching → geometric estimation → warping → blending. No training, no data, no neural networks. Just mathematics, geometry, and careful algorithm design.

## Summary: The Classical CV Pipeline

| Step | Technique | Era | What it does | Key Math |
|------|-----------|-----|-------------|----------|
| 1 | Image as array | 1960s | Pixels = numbers | Sampling, quantization, Nyquist |
| 2 | Color spaces (HSV, Lab) | 1970s | Separate brightness from color | Illumination-reflectance decomposition |
| 3 | Edge detection (Sobel/Canny) | 1960s-80s | Find boundaries | Gaussian derivative, NMS, hysteresis |
| 4 | Corner detection (Harris) | 1988 | Find distinctive points | Structure matrix, eigenvalue analysis |
| 5 | Contours & shapes | 1985 | Outline and classify objects | Chain codes, Douglas-Peucker |
| 6 | Feature matching (ORB/SIFT) | 1999-2006 | Match points across images | Scale-space, ratio test, Hamming distance |
| 7 | Image stitching (homography) | 2000s | Combine images into panoramas | RANSAC, projective geometry |

> **The through-line:** Each step adds a layer of *understanding*. Pixels → edges → corners → shapes → matches → geometric transformation. Each layer abstracts away irrelevant detail and preserves what matters.

> **The bridge to deep learning:** Modern neural networks do exactly the same thing, but automatically. CNN first layers learn Sobel-like edges. Second layers learn corner-like patterns. Deeper layers learn shape-like features. **Deep learning didn't replace classical CV — it automated it.** The mathematical operations are identical; only the method of determining the kernel weights differs (hand-crafted vs. learned via gradient descent).

> **The bridge to modern AI:** Transformers (ViT) replaced local convolution with global attention. Diffusion models replaced classification with iterative denoising. JEPA replaces pixel prediction with latent-space prediction. Each step automates another aspect of human intelligence — from feature extraction to world modeling.

## 8. A Small CNN with Flax NNX — Classical vs. AI Workflows

Now let's build a neural network from scratch using **Flax NNX** (JAX's neural network library). We'll train a small CNN to classify our synthetic shapes.

> **The workflow difference:**
> - **Classical CV:** Engineer features (edges, corners, descriptors) → hand-craft pipeline → geometric matching
> - **AI/CV:** Define architecture → feed data → let optimization learn the features automatically

The CNN we'll build is simple but captures the key concepts: convolutional layers, pooling, activation functions, and classification heads.

### 8.1 The Convolutional Layer: Learned Filters

A convolutional layer is **exactly the same operation** as the Sobel/Gaussian kernels from Section 3 — but instead of being hand-crafted, the kernel weights are **learned parameters**.

For a conv layer with $C_{\text{in}}$ input channels, $C_{\text{out}}$ output channels, and kernel size $k \times k$:

$$\text{Output}_{c}(i,j) = \sum_{c'=0}^{C_{\text{in}}-1} \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} W_{c,c',m,n} \cdot \text{Input}_{c'}(i+m, j+n) + b_c$$

**Parameter count:** $C_{\text{in}} \times C_{\text{out}} \times k \times k + C_{\text{out}}$ (biases). For our first layer (3→16, 3×3): $3 \times 16 \times 3 \times 3 + 16 = 448$ parameters. These are learned via backpropagation.

### 8.2 Backpropagation Through a Conv Layer

The training loop uses **gradient descent**:

1. **Forward pass:** Compute output = Conv(input, W)
2. **Loss computation:** Compare output to ground truth (cross-entropy)
3. **Backward pass:** Compute $\frac{\partial \mathcal{L}}{\partial W}$ using the chain rule
4. **Update:** $W \leftarrow W - \eta \cdot \frac{\partial \mathcal{L}}{\partial W}$

The gradient flows backward through the convolution, which is itself a convolution with the input (flipped). This is the **core insight of CNNs**: the same convolution operation works in both directions.

### 8.3 Activation Functions: Introducing Non-Linearity

Without activation functions, a stack of conv layers is just a **linear transformation** (composition of linear operations is linear). ReLU ($f(x) = \max(0, x)$) introduces non-linearity:

- **Why ReLU over sigmoid?** Sigmoid saturates for large positive/negative inputs (gradient ≈ 0), causing the **vanishing gradient** problem. ReLU has gradient = 1 for all positive inputs, allowing gradients to flow through many layers.
- **Dead ReLU problem:** Neurons with negative initial weights can get "killed" (always output 0). In practice, this is rare with proper initialization.

In [ ]:
# Flax NNX imports (Flax 0.8+)
import flax.nnx as nnx
import jax
import jax.numpy as jnp
from jax import random

# Print device info
print(f"Devices: {jax.devices()}")


### CNN Architecture: From Pixels to Classes

Our network:

$$\text{Conv(3→16, 3×3)} \xrightarrow{\text{ReLU}} \text{AvgPool(2×2)} \xrightarrow{}
\text{Conv(16→32, 3×3)} \xrightarrow{\text{ReLU}} \text{AvgPool(2×2)} \xrightarrow{}
\text{Flatten} \xrightarrow{} \text{Dense(32×16×16→5)} \xrightarrow{} \text{Classes}$$

**Spatial dimensions through the network** (input: 64×64):

| Layer | Output size | What it computes |
|-------|------------|------------------|
| Input | 64 × 64 × 3 | Raw pixels |
| Conv1 + ReLU | 64 × 64 × 16 | Edge/corner detectors (learned) |
| AvgPool | 32 × 32 × 16 | Spatial downsampling (2× reduction) |
| Conv2 + ReLU | 32 × 32 × 32 | Higher-level patterns (combinations of edges) |
| AvgPool | 16 × 16 × 32 | Further downsampling |
| Flatten | 8192 | Vector of all feature maps |
| Dense | 5 | Classification logits |

**This mirrors what deeper networks like ResNet do**, just much smaller:
- **Conv layers** learn edge/corner/shape features (like classical CV, but learned from data)
- **Pooling** provides spatial invariance — the exact position of a feature doesn't matter, only its presence
- **Dense layers** classify based on the presence/absence of learned features

**Receptive field growth:** After the first conv layer, each output pixel depends on a 3×3 region of the input. After pooling + conv + pooling, each output pixel depends on a 15×15 region of the original input. Deeper networks have **larger receptive fields** — they can "see" more of the image.

In [ ]:
class SimpleCNN(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.conv1 = nn.Conv(in_channels=3, out_channels=16, kernel_size=3)
        self.conv2 = nn.Conv(in_channels=16, out_channels=32, kernel_size=3)
        self.fc1 = nn.Linear(in_features=32 * 32 * 32, out_features=5)
        self.relu = nnx.ReLU

    def __call__(self, x):
        x = self.relu(self.conv1(x))
        x = nnx.avg_pool(x, window_size=(2, 2), strides=(2, 2))  # 64->32
        x = self.relu(self.conv2(x))
        x = nnx.avg_pool(x, window_size=(2, 2), strides=(2, 2))  # 32->16
        x = x.reshape(x.shape[0], -1)  # flatten
        x = self.fc1(x)
        return x

# Create model with RNGs for initialization
rng = random.key(42)
model = SimpleCNN(nnx.Rngs(rng))
optimizer = nnx.Optimizer(model)

n_params = sum(p.size for p in nnx.state(model).values())
print(f"CNN created. Parameters: {n_params:,}")


### Preparing Data

We'll create simple synthetic images for each class:
- **Class 0:** Red circles
- **Class 1:** Green squares
- **Class 2:** Blue triangles
- **Class 3:** Yellow rectangles
- **Class 4:** Mixed shapes

In [ ]:
def make_dataset(rng, n_per_class=100, size=64):
    """Generate synthetic classification dataset."""
    images, labels = [], []
    for cls in range(5):
        for _ in range(n_per_class):
            img = np.zeros((size, size, 3), dtype=np.uint8)
            colors = [
                (0, 0, 255),   # Red (BGR)
                (0, 255, 0),   # Green
                (255, 0, 0),   # Blue
                (0, 255, 255), # Yellow
                (255, 255, 255), # White
            ]
            color = colors[cls]
            cx, cy = np.random.randint(15, size-15, 2)
            if cls == 0:  # Circle
                cv2.circle(img, (cx, cy), np.random.randint(10, 20), color, -1)
            elif cls == 1:  # Square
                s = np.random.randint(15, 30)
                cv2.rectangle(img, (cx-s//2, cy-s//2), (cx+s//2, cy+s//2), color, -1)
            elif cls == 2:  # Triangle
                pts = np.array([[cx, cy-15], [cx-15, cy+15], [cx+15, cy+15]], np.int32).reshape((-1,1,2))
                cv2.fillPoly(img, [pts], color)
            elif cls == 3:  # Rectangle
                w, h = np.random.randint(20, 35), np.random.randint(10, 15)
                cv2.rectangle(img, (cx-w//2, cy-h//2), (cx+w//2, cy+h//2), color, -1)
            else:  # Mixed
                cv2.circle(img, (cx-10, cy), 8, color, -1)
                cv2.rectangle(img, (cx+5, cy-8), (cx+15, cy+8), color, -1)
            # Add some noise
            noise = np.random.randint(0, 20, img.shape, dtype=np.uint8)
            img = cv2.bitwise_or(img, noise)
            images.append(img)
            labels.append(cls)
    images = np.array(images, dtype=np.float32) / 255.0
    labels = np.array(labels)
    return images, labels

X, y = make_dataset(rng)
print(f"Dataset: {X.shape} images, {y.shape} labels")
print(f"Class distribution: {np.bincount(y)}")


### Training Loop: Gradient Descent in JAX

Standard mini-batch training with cross-entropy loss. The key JAX pattern:
- `jax.grad` computes gradients using **automatic differentiation** (reverse-mode autodiff, also known as backpropagation)
- Gradients update the optimizer
- `jax.jit` compiles the training step for speed using XLA (Accelerated Linear Algebra)

**Cross-entropy loss:** For a single sample with true class $c$ and logits $z_1, \ldots, z_K$:

$$\mathcal{L} = -\log\left(\frac{e^{z_c}}{\sum_{k=1}^{K} e^{z_k}}\right) = -z_c + \log\left(\sum_{k=1}^{K} e^{z_k}\right)$$

The gradient with respect to the logits is simply the **difference between the prediction and the truth**: $\frac{\partial \mathcal{L}}{\partial z_k} = p_k - \mathbb{1}(k = c)$, where $p_k$ is the softmax probability. This is a beautiful result: the gradient points in the direction of increasing prediction error.

**JAX's `value_and_grad`:** Computes both the loss and its gradients in a single pass. This is more efficient than computing them separately because the gradient computation can reuse the forward pass intermediates.

In [ ]:
@jax.jit
def train_step(optimizer, images, labels):
    def loss_fn(model):
        logits = model(images)
        log_softmax = logits - jax.nn.log_softmax(logits)
        loss = jnp.mean(-log_softmax[jnp.arange(len(labels)), labels])
        return loss
    loss, grads = jax.value_and_grad(loss_fn)(optimizer)
    optimizer.update(grads)
    return loss

@jax.jit
def evaluate(model, images, labels):
    logits = model(images)
    preds = jnp.argmax(logits, axis=1)
    return jnp.mean(preds == labels)

# Train
batch_size = 32
epochs = 10
n = X.shape[0]

for epoch in range(epochs):
    perm = random.permutation(rng, n)
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        train_step(optimizer, X[idx], y[idx])
    acc = evaluate(model, X, y)
    print(f"Epoch {epoch+1:2d}/{epochs} | Accuracy: {acc:.1%}")


### What Did the CNN Learn? — Connecting Back to Classical CV

Compare with classical CV:

| Aspect | Classical CV (Sobel/Canny) | CNN (Flax NNX) |
|--------|---------------------------|-----------------|
| Features | Hand-crafted kernels | Learned from data |
| Pipeline | Edge → Corner → Match → Geometry | Raw pixels → Classes |
| Generalization | Limited to designed invariances | Learned from examples |
| Interpretability | High (we know exactly what each step does) | Lower (black box) |
| Data needed | None | Hundreds of examples |
| Computational cost | CPU, real-time | GPU, training time |

**The visualization insight:** The CNN's first conv layer weights, when visualized, look remarkably like edge detectors and color blobs — similar to Sobel/Gabor filters. **The network learned classical CV features automatically.**

**Why does this happen?** The gradient descent optimization discovers that edges and corners are the most useful features for distinguishing shapes. The convolutional architecture **biases** the network toward learning spatially-local features (the **inductive bias** of convolution). Combined with the gradient signal from the loss, the first layer naturally converges to edge-like detectors.

**The hierarchy of features:** In deeper networks:
- **Layer 1:** Edges, corners, color blobs (like Sobel/Gabor)
- **Layer 2:** Combinations of edges → corners at different angles, simple textures
- **Layer 3+:** Shape parts (circles, rectangles, triangles)
- **Final layers:** Full object parts or whole objects

This hierarchy was **confirmed by visualizations** of AlexNet's first layer (Krizhevsky et al., 2012) and later work (Zeiler & Fergus, 2014) — deeper layers reconstruct increasingly complex visual concepts. **Deep learning didn't replace classical CV — it automated it.**

## 9. AlexNet & ResNet — The Deep Learning Revolution

### 9.1 AlexNet (2012) — The ImageNet Breakthrough

Alex Krizhevsky's AlexNet won the ImageNet Competition with a **16.4% top-5 error** vs. 26.2% for the runner-up (a classical CV system). This 36% relative error reduction is considered the **turning point** where deep learning became dominant in CV.

**Key innovations and why they worked:**

1. **Deep architecture (8 layers, 60M parameters):** Previous networks were too shallow and small. AlexNet showed that **depth + capacity** matters when you have enough data and compute. The hierarchical feature learning (edges → textures → parts → objects) requires many layers to build up.

2. **ReLU activations:** ReLU ($f(x) = \max(0, x)$) trains **5× faster** than sigmoid because it doesn't saturate for positive inputs. The vanishing gradient problem (sigmoid's derivative is at most 0.25) made deep networks impractical with sigmoid.

3. **Dropout (0.5 probability):** Randomly zero out 50% of neurons during training. This prevents **co-adaptation** — neurons can't rely on specific other neurons being present. Effectively trains an exponential number of overlapping sub-networks. At test time, scale all weights by 0.5.

4. **Data augmentation:** Rotate and crop images during training. This artificially expands the dataset and teaches the network **translation and rotation invariance**. Without this, the network overfits to the training set.

5. **GPU training:** Used two NVIDIA GTX 580 GPUs (3GB VRAM each). Convolution on GPUs is **10-20× faster** than CPU because each pixel/filter operation is parallelizable. This was the first demonstration that GPUs make deep CV practical.

### 9.2 The Vanishing Gradient Problem

As networks got deeper (10+, 20+ layers), a new problem emerged: **accuracy saturated and then degraded**. Deeper networks were *worse*, not better.

**The math:** In backpropagation, the gradient of the loss with respect to early layer weights involves the product of many Jacobians:

$$\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial W_L} \cdot \frac{\partial W_L}{\partial W_{L-1}} \cdot \ldots \cdot \frac{\partial W_2}{\partial W_1}$$

If each Jacobian has singular values less than 1 (typical with sigmoid/tanh), the product **exponentially decays** to zero. Gradients vanish before reaching early layers. Early layers stop learning.

**Batch Normalization** (Ioffe & Szegedy, 2015) partially mitigates this by normalizing layer inputs to have zero mean and unit variance, keeping activations in the linear regime of the activation function.

### 9.3 ResNet (2015) — Solving Degradation with Skip Connections

Kaiming He's ResNet (Residual Network) solved the degradation problem with **skip connections** (also called residual connections or shortcut connections):

$$\text{Output} = \text{Activation}(\mathcal{F}(x) + x)$$

where $\mathcal{F}(x)$ is the residual function (a stack of layers) and $+x$ is the **skip connection** that bypasses the stack.

**Why this works:**
- **Gradient flow:** The skip connection provides a **direct path** for gradients. During backprop: $\frac{\partial \text{Output}}{\partial x} = \frac{\partial \mathcal{F}}{\partial x} + I$. The identity matrix $I$ ensures gradients never vanish completely.
- **Easier optimization:** Learning the identity mapping is easier than learning a full transformation. If the layers don't need to change anything, they can just set weights to zero and pass $x$ through.
- **Residual learning:** The network learns $\mathcal{F}(x) = \text{Target} - x$ (the residual) rather than the full target. Residuals are smaller and easier to learn.

**The impact:** ResNet-152 achieved 3.57% top-5 error on ImageNet (below human-level, ~5.25%). This was the first CV system to outperform humans on this benchmark.

In [ ]:
# Load pretrained models with ImageNet classes
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import torch

# Load ImageNet-1K label mapping (1000 classes)
try:
    from timm.data._imagenet_classes import _class_names
    imagenet_labels = _class_names
except ImportError:
    # Fallback: generate placeholder labels
    imagenet_labels = [f"class_{i}" for i in range(1000)]

# Load pretrained ResNet-18 (classic CNN architecture)
# Note: timm doesn't include classic AlexNet (2012) architecture.
# ResNet-18 represents the "classic CNN" era (2012-2015).
print("Loading pretrained ResNet-18 (ImageNet-1K)...")
model = timm.create_model("resnet18", pretrained=True, num_classes=1000)
model.eval()

config = resolve_data_config({}, model=model)
transform = create_transform(**config)

print(f"Model: resnet18")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"ImageNet-1K classes loaded: {len(imagenet_labels)}")


In [ ]:
# Run inference on our synthetic image
img = torch.tensor(transform(color_scene)).unsqueeze(0)

with torch.no_grad():
    logits = model(img)
    probs = torch.softmax(logits, dim=1)[0]

# Top 5 predictions with ImageNet labels
top5_idx = probs.topk(5).indices.tolist()
top5_probs = probs.topk(5).values.tolist()

print("\nTop 5 ImageNet predictions:")
for idx, prob in zip(top5_idx, top5_probs):
    label = imagenet_labels[idx] if idx < len(imagenet_labels) else f"class_{idx}"
    print(f"  [{prob:.2%}] {label}")

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(color_scene)
label = imagenet_labels[top5_idx[0]] if top5_idx[0] < len(imagenet_labels) else f"class_{top5_idx[0]}"
plt.title(f"Top: {label} ({top5_probs[0]:.2%})")
plt.axis('off')
plt.tight_layout()
plt.show()


### Visualizing What CNNs See: First-Layer Filter Analysis

The first layer of a CNN learns filters that respond to specific patterns. For AlexNet and early CNNs, these look like:
- **Edges** at various orientations (like Sobel)
- **Color blobs** (like the color space separation we did)
- **Gabor-like patterns** (orientation + frequency selective)

**The mathematical connection:** The first-layer filters solve the optimization problem:

$$\min_W \mathbb{E}_{\text{data}}[\mathcal{L}(W, \text{data})]$$

Subject to the constraint that $W$ are convolution kernels. The gradient signal from the classification loss pushes filters toward features that are **discriminative** for the task. For ImageNet, the most discriminative low-level features are edges and color blobs — exactly what classical CV hand-crafted.

**But there's a crucial difference:** Classical CV filters are **universal** (Sobel works on any image). CNN filters are **task-specific** — they learn the edges and patterns that are most useful for distinguishing ImageNet classes. Different tasks produce different first-layer filters.

This is the "aha" moment: **deep learning automated classical CV.** The network learned edge detectors, color separators, and pattern recognizers — all from data, no hand-crafting required. The mathematical operations are identical; only the method of determining the kernel weights differs.

In [ ]:
# Extract first conv layer weights
# ResNet has conv1; ViT has patch_embed.proj
if hasattr(model, 'conv1'):
    weights = model.conv1.weight
elif hasattr(model, 'patch_embed') and hasattr(model.patch_embed, 'proj'):
    weights = model.patch_embed.proj.weight
else:
    weights = list(model.parameters())[0]

weights = weights.cpu()
if weights.ndim == 4:
    # Average across input channels for visualization
    weights = weights.mean(dim=1)

# Normalize and display first 16 filters
n = min(16, weights.shape[0])
fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for i in range(n):
    w = weights[i]
    w_min, w_max = w.min(), w.max()
    if w_max - w_min < 1e-8:
        w_display = np.zeros_like(w.numpy())
    else:
        w_display = ((w - w_min) / (w_max - w_min) * 255).astype(np.uint8)
    axes[i // 4, i % 4].imshow(w_display, cmap='gray')
    axes[i // 4, i % 4].axis('off')

axes[0, 0].set_title("First Layer Filters (learned edges/patterns)")
plt.tight_layout()
plt.show()


## 10. Vision Transformer (ViT) — Images as Sequences (2020)

Transformers dominated NLP. Could they work for images?

> **Historical note:** The Vision Transformer (Dosovitskiy et al., "An Image is Worth 16×16 Words", 2020) showed that transformers could match or beat CNNs on image classification — **if trained on enough data**. The key insight: treat image patches as tokens in a sequence, exactly like words in a sentence. This bridged the gap between computer vision and the transformer architecture that would power LLMs.

### 10.1 CNN vs. ViT: Inductive Biases

Every architecture encodes **assumptions about the data** (inductive biases):

| Aspect | CNN (ResNet) | ViT |
|--------|-------------|-----|
| **Inductive bias** | Local connectivity, translation equivariance | None (learned globally) |
| **Receptive field** | Grows with depth (local → global) | Global from layer 1 |
| **Data hunger** | Moderate (pretraining helps) | High (needs lots of data) |
| **Parallelization** | Sequential layers (layer-by-layer) | Fully parallel (like LLMs) |
| **Parameter efficiency** | High (weight sharing in conv) | Lower (no weight sharing) |

**The trade-off:** CNNs' inductive bias (local connectivity) means they learn faster from less data but may be less flexible. ViT has no such bias — it can learn **any** relationship from pixel $i$ to pixel $j$, but needs vastly more data to discover these relationships on its own.

**Key finding (Dosovitskiy et al.):** ViT matches or exceeds ResNet **only when trained on large datasets** (ImageNet-21K, JFT-300M). On small datasets (ImageNet-1K with 1K samples), ResNet significantly outperforms ViT. This confirms that the inductive bias of convolution matters most when data is limited.

### 10.2 How ViT Works: Patch-Based Processing

1. **Patchify:** Split image into fixed-size patches (e.g., 16×16 pixels). A 224×224 image becomes 196 patches (14 × 14). Each patch is a vector of $16 \times 16 \times 3 = 768$ values.

2. **Linear embed:** Project each patch to a $d$-dimensional vector using a learned linear transformation. For ViT-Base: $d = 768$. This is equivalent to a convolution with stride = patch_size and no non-linearity.

3. **Add position embeddings:** Patches lose spatial info when flattened. Add learned positional encodings to retain spatial structure. Without these, ViT is **permutation-invariant** (treating patches as a set, not a sequence).

4. **Transformer encoder:** Standard multi-head self-attention across all patches. Each patch attends to every other patch. $L$ layers of: Multi-Head Attention → Add & Norm → MLP → Add & Norm.

5. **Classification head:** Use the [CLS] token output (a special learnable token prepended to the sequence) → linear layer → classes.

### 10.3 Multi-Head Self-Attention: The Core Mechanism

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

where each head is:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**The attention map as a "soft" correspondence mechanism:** The attention matrix $\text{softmax}(QK^T/\sqrt{d_k})$ is a $N \times N$ matrix (where $N$ is the number of patches). Each row gives a **probability distribution** over all patches for a given patch. This is analogous to the feature matching in Section 6 — but **learned**, not computed from hand-crafted descriptors.

**Key difference from CNN:** In a CNN, pixel $(100, 100)$ can only "see" its local neighborhood (e.g., 3×3 or 7×7). To understand the whole image, information must propagate through many layers (logarithmic in image size). In ViT, **every patch attends to every other patch** from layer 1 — constant-time access to the entire image.

In [ ]:
# Load a ViT model
vit = timm.create_model("vit_base_patch16_224.augreg_in21k_ft1k", pretrained=True, num_classes=1000)
vit.eval()

vit_config = resolve_data_config({}, model=vit)
vit_transform = create_transform(**vit_config)

print(f"ViT Model: {vit.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in vit.parameters()):,}")
print(f"Patch size: 16x16 -> {224//16}x{224//16} = {(224//16)**2} patches per image")

# Run inference
img_vit = torch.tensor(vit_transform(color_scene)).unsqueeze(0)
with torch.no_grad():
    logits_vit = vit(img_vit)
    probs_vit = torch.softmax(logits_vit, dim=1)[0]

top5_vit = probs_vit.topk(5)
print("\nViT Top 5 predictions:")
for idx, prob in zip(top5_vit.indices.tolist(), top5_vit.values.tolist()):
    print(f"  [{prob:.2%}] Class {idx}")


### Key Insight: Self-Attention as Global Context

In a CNN, a pixel at position (100, 100) can only "see" its local neighborhood (e.g., 3×3 or 7×7). To understand the whole image, information must propagate through many layers.

In a ViT, **every patch attends to every other patch** from layer 1. The self-attention mechanism:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

This means the ViT can immediately relate the top-left corner to the bottom-right corner — no propagation delay. But it also means no built-in assumption that nearby pixels are more related (the CNN's inductive bias).

**Computational cost:** Attention is $O(N^2 \cdot d)$ where $N$ is the number of patches and $d$ is the embedding dimension. For a 224×224 image with 16×16 patches: $N = 196$, so the attention matrix is 196 × 196 = 38,416 entries per head. This is manageable, but for larger images, the quadratic cost becomes prohibitive. CNNs, by contrast, are $O(N)$ in the number of pixels (each pixel is processed independently with a fixed-size kernel).

**Visualizing attention:** Research (Abnar & Zuidema, 2020; Vig, 2019) shows that ViT attention heads often specialize: some attend to local edges, some to object parts, some to global structure. This is analogous to the **hierarchy of features** in CNNs — but learned through attention rather than through stacked convolutions.

## 11. Diffusion Models — Learning to Dream (2020–present)

After GANs (2014) promised perfect generation but suffered from mode collapse and training instability, diffusion models emerged as the stable alternative.

> **Historical note:**
> - **2014:** Goodfellow introduces GANs — generator vs. discriminator battle. Theoretically elegant but practically fragile.
> - **2015:** Sohl-Dickstein et al. introduce first diffusion models (deep generative models via nonequilibrium thermodynamics). The connection to physical processes (heat diffusion) provided a principled framework.
> - **2020:** DDPM (Ho et al., "Denoising Diffusion Probabilistic Models") makes diffusion practical and scalable. Simple architecture, competitive with GANs.
> - **2021:** Stable Diffusion (Rombach et al.) combines diffusion with **latent space** — perform the diffusion process in a compressed representation learned by an autoencoder. This makes diffusion feasible for high-resolution images.
> - **2022:** DALL-E 2, Midjourney, Stable Diffusion 2 — diffusion-powered text-to-image generation captures public imagination.

### 11.1 The Forward Process: Markovian Noising

The forward process gradually adds Gaussian noise over $T$ timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} \cdot x_{t-1}, \beta_t \cdot I)$$

where $\beta_t$ is a **variance schedule** (typically linear or cosine, from $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$).

**Key trick — closed-form sampling:** By repeatedly applying the chain rule, we can sample $x_t$ directly from $x_0$ without iterating through all $t$ steps:

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1-\bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$. This allows us to train with **any random timestep** $t$ — we don't need to simulate the full forward process.

After $T$ steps (typically 1000), $x_T \approx \mathcal{N}(0, I)$ — pure Gaussian noise.

### 11.2 The Reverse Process: Learning to Denoise

The reverse process is a Markov chain parameterized by a neural network:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 \cdot I)$$

The network predicts the **mean** $\mu_\theta$ (the denoised version of $x_t$). Two common parameterizations:

1. **Predict noise** ($\epsilon$-parameterization, DDPM): The network predicts $\epsilon_\theta(x_t, t)$ — the noise that was added at timestep $t$. The denoised image is recovered as:

$$\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \cdot \epsilon_\theta(x_t, t)\right)$$

2. **Predict $x_0$** ($x_0$-parameterization): The network directly predicts the original image at timestep $t$.

**The loss function** (simplified DDPM loss):

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

This is a **weighted MSE loss** — the network learns to predict the noise at a random timestep. The weighting can be adjusted to emphasize different timesteps (simple weighting vs. SNR weighting).

### 11.3 Why Diffusion Works (Where GANs Failed)

| Property | GAN | Diffusion |
|----------|-----|-----------|
| **Training stability** | Unstable (minimax game, mode collapse) | Stable (simple MSE regression) |
| **Mode collapse** | Common (generator finds one mode and exploits it) | Rare (each step is independent regression) |
| **Generation speed** | Fast (1 step: noise → image) | Slow (50-1000 steps: iterative denoising) |
| **Sample quality** | Can be very sharp | Slightly softer (averaging effect of many steps) |
| **Diversity** | Poor (mode collapse) | Excellent (covers full distribution) |
| **Likelihood** | Intractable | Lower-bounded (variational lower bound) |

**The key insight:** Diffusion turns generation into a series of **simple denoising steps**. Instead of learning to generate a perfect image in one shot (GAN's single function approximation), it learns to remove a little noise at a time. Each step is easy (predicting noise is simpler than generating pixels), and the composition is powerful.

**Mathematical connection:** Diffusion models maximize a **variational lower bound** on the data likelihood (Sohl-Dickstein et al., 2015; Sohl-Dickstein & Weiss, 2015). The forward process defines a fixed distribution $q$, and the reverse process learns to invert it. This is equivalent to a **score matching** objective (Hyvärinen, 2005): learn the gradient of the log-density (the "score") at each noise level.

In [ ]:
# DDPM inference with diffusers
from diffusers import DDIMPipeline
import torch

print("Loading DDPM pipeline (first run downloads model weights ~2GB)...")
pipe = DDIMPipeline.from_pretrained("google/ddpm-celebahq-256")
# Alternative smaller models:
# pipe = DDIMPipeline.from_pretrained("google/ddpm-bedroom-256")
# pipe = DDIMPipeline.from_pretrained("google/ddpm-cat-256")

# Generate an image from noise
print("Generating image from noise (this may take a minute)...")
torch_rng = torch.Generator().manual_seed(42)
image = pipe(num_inference_steps=50, generator=torch_rng)
generated_img = image.images[0]

plt.figure(figsize=(8, 8))
plt.imshow(generated_img)
plt.title("Generated from Noise via DDPM")
plt.axis('off')
plt.tight_layout()
plt.show()


### Why Diffusion Works (Where GANs Failed) — A Deeper Look

| Property | GAN | Diffusion |
|----------|-----|-----------|
| Training stability | Unstable (minimax game) | Stable (simple denoising MSE) |
| Mode collapse | Common | Rare |
| Generation speed | Fast (1 step) | Slow (50-1000 steps) |
| Sample quality | Can be sharp | Slightly softer |
| Diversity | Poor (mode collapse) | Excellent |

**The fundamental difference:** A GAN learns a **single function** $G: z \to x$ (noise → image). This is a highly complex, multi-modal function that maps a simple distribution to the image distribution. The discriminator provides a gradient signal, but it's sparse and unstable.

A diffusion model learns **1000 simpler functions** $D_t: (x_t, t) \to \epsilon$ (denoise at step $t$). Each function is a simple regression task: "remove a little noise." The composition of 1000 simple functions is complex, but each individual function is easy.

**The thermodynamics analogy:** The forward process is like dropping ink into water — the ink diffuses and you can't recover it. The reverse process is like reversing the heat equation — mathematically ill-posed, but with a neural network prior, it becomes feasible. The noise schedule acts as a **cooling schedule** (analogous to simulated annealing), gradually refining the sample from coarse to fine structure.

## 12. JEPA — Predicting in Latent Space (2022–present)

### 12.1 The Problem with Pixel/Token Prediction

LLMs predict the next token. Diffusion predicts pixels. Both operate at the **surface level** of data:
- They try to reproduce every detail, including unpredictable noise
- They waste model capacity on details that don't matter for understanding
- They can't reason about abstract concepts or counterfactuals

**Example:** If you ask an LLM to complete "The sky is...", it predicts "blue" because that's the most common continuation. But it can't reason about "The sky is blue **because**..." — the causal structure behind the observation. Similarly, a diffusion model trained to reconstruct pixels will learn the texture of grass but not the concept of "grass" as a ground surface.

> **Yann LeCun's argument (2022):** To build true AI, we need models that learn **internal models of the world** — not just memorize surface patterns. A model that predicts pixels is a **stochastic parrot**. A model that predicts abstract representations is building a **world model**.

### 12.2 JEPA: Joint Embedding Predictive Architecture

Proposed by Yann LeCun in "A Path Towards Autonomous Machine Intelligence" (Feb 2022).

**Core idea:** Instead of predicting pixels or tokens, predict **abstract representations (embeddings)** in latent space. The key insight: the future is **deterministic at the level of concepts** but **stochastic at the level of pixels**. The same scene (a cat sitting on a mat) can look completely different under different lighting, angles, and camera qualities — but the **abstract representation** (cat + mat + spatial relationship) remains stable.

### 12.3 How JEPA Works: The Prediction Framework

1. **Encode context region** $x$ → embedding $s_x = E_x(x)$
2. **Encode target region** $y$ → embedding $s_y = E_y(y)$ (with **stop-gradient** on $E_y$)
3. **Predict target embedding** from context: $\hat{s}_y = P_\phi(s_x)$
4. **Loss:** $L = \|\hat{s}_y - s_y\|^2$ (L2 distance in latent space)

**The stop-gradient is critical:** Without it, the predictor $P_\phi$ can simply copy $s_x$ to minimize the loss (trivial solution). By blocking the gradient through $E_y$, the predictor is forced to **learn the relationship** between context and target, not just memorize the encoder output.

**Why L2 in latent space?** In pixel space, L2 measures pixel-wise similarity (which is meaningless for high-frequency details). In latent space, L2 measures **semantic similarity** — are the two representations close in the space of meaning?

### 12.4 Preventing Representation Collapse

A fundamental challenge in self-supervised learning: the encoder can collapse all inputs to the **same representation** (trivial solution: $E(x) = c$ for all $x$). If all embeddings are identical, the predictor's job is trivial (predict $c$ from $c$).

**Solutions in JEPA:**

1. **Stop-gradient on target encoder:** Prevents the predictor from bypassing the learning task.

2. **EMA (Exponential Moving Average) updates:** The target encoder's weights are updated as an exponential moving average of the online encoder's weights:

$$\bar{\theta} \leftarrow \tau \bar{\theta} + (1-\tau) \theta$$

where $\tau \approx 0.999$. This creates a **slowly evolving target** that provides stable learning signals. The EMA acts as a form of temporal consistency regularization.

3. **Multiple latent variables:** Predict multiple target regions from the same context. This forces the encoder to capture **diverse** aspects of the scene, not just one collapsed representation.

### 12.5 JEPA vs. Alternative Self-Supervised Approaches

| Method | Predicts | Space | Needs Negatives? | What it learns |
|--------|----------|-------|------------------|----------------|
| Pixel reconstruction | Raw pixels | Pixel space | No | Texture, high-frequency detail |
| Contrastive learning (SimCLR) | Same embedding for augmentations | Embedding space | **Yes** (InfoNCE loss) | Invariant representations |
| **JEPA** | **Target embedding from context** | **Latent space** | **No** | **Causal structure, world model** |
| LLM | Next token | Token space | No | Language patterns |
| Diffusion | Next denoised image | Pixel space | No | Image distribution |

**Key distinction from contrastive learning:** Contrastive learning requires **negative samples** (many different images in a batch) to distinguish "same" from "different." JEPA doesn't need negatives — it learns from **spatial or temporal structure** in the data itself. The context-target relationship provides the supervisory signal without requiring negative examples.

### 12.6 The JEPA Family

- **I-JEPA** (Jun 2023): Images — predict masked patch representations. Shows that predicting latent representations of spatially masked regions teaches meaningful visual concepts.

- **MC-JEPA** (Jul 2023): Motion + Content — joint learning of optical flow and static content. The predictor learns both **what** is in the scene and **how** it moves.

- **V-JEPA** (Feb 2024): Video — spatio-temporal prediction. Predicts future latent representations from past frames. Learns **temporal dynamics** without seeing pixels.

- **V-JEPA 2** (Jun 2025): Video + Robotics — world models for planning. Uses learned latent dynamics for **action planning** in robotic manipulation.

- **VL-JEPA**: Vision-Language — predict meaning, then decode text. Bridges visual understanding with language.

### 12.7 Why JEPA Matters for the Future of AI

1. **Compute efficiency:** No pixel reconstruction overhead. The model focuses capacity on **semantically relevant** predictions, not high-frequency noise.

2. **Semantic learning:** Predicting in latent space forces the model to learn **meaning**, not texture. The encoder discards unpredictable detail; the predictor learns causal structure.

3. **No negative samples:** Unlike contrastive learning, no need for huge batches of negative examples. This makes JEPA more scalable and easier to train.

4. **World models:** The predictor IS a primitive **world model** — it models spatial/temporal uncertainty. Given a context, it can predict multiple possible futures (by sampling from the distribution of predictions).

5. **Planning:** Predict in latent space → simulate futures efficiently → choose actions that lead to desired outcomes. This is the foundation of **model-based reinforcement learning** at scale.

> **The big picture:** JEPA represents a shift from "predict everything" to "predict what matters." The encoder discards unpredictable detail; the predictor learns causal structure. This is closer to how humans learn — we remember the **structure of events**, not every pixel of every scene. We can imagine "what if" scenarios in our heads without rendering them in photorealistic detail. JEPA is a step toward that kind of **abstract reasoning**.

## Complete CV Timeline

| Era | Year | Milestone | Key Idea |
|-----|------|-----------|----------|
| **Foundations** | 1957 | First digital photo (Kirsch) | Image = pixel grid |
| | 1930s | CIE color spaces | Human psychophysics |
| **Classical CV** | 1960s | RETINEX (Land) | Edges > absolute brightness |
| | 1983 | Marr-Hildreth | Zero-crossings of Laplacian |
| | 1985 | Suzuki contours | Topological hierarchy |
| | 1986 | Canny edge detector | Optimal edge detection (3 criteria) |
| | 1988 | Harris corner detector | Structure matrix eigenvalues |
| | 1999 | SIFT (Lowe) | Scale-invariant features |
| | 2006 | ORB | Fast, patent-free features |
| | 2000s | Image stitching | Classical CV capstone |
| **Deep Learning** | 2012 | AlexNet | Hierarchical feature learning, ReLU, dropout |
| | 2015 | ResNet | Skip connections, 152 layers, below human |
| | 2020 | ViT | Images as sequences, self-attention |
| **Generation** | 2014 | GANs (Goodfellow) | Adversarial generation |
| | 2020 | DDPM (Ho et al.) | Stable diffusion, score matching |
| | 2021 | Stable Diffusion | Latent diffusion |
| **Understanding** | 2022 | JEPA (LeCun) | Predict in latent space |
| | 2023-25 | V-JEPA, VL-JEPA | World models for robotics |

> **Final thought:** We started with Sobel filters and ended with models that predict in latent space. But the Sobel filter is still in every CNN first layer — just learned instead of hand-crafted. The story of computer vision is not replacement, but **automation and abstraction**: each new era builds on the insights of the previous one, scaling them up and automating the hand-crafting. The mathematics evolves (gradients → attention → score matching → latent prediction), but the core question remains the same: **how do we extract meaning from light?**

---

*End of Classical Computer Vision section.*